In [4]:
# Шаг 1. Создание структуры папок, анализ дисбаланса и обучение модели
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score
import joblib

# Указываем точный путь к проекту на Google Диске
project_name = "/content/drive/MyDrive/Ucheba/Input ML/credit-card-ml-deployment"

# 1. Создаем структуру папок
folders_to_create = [
    f"{project_name}/app",
    f"{project_name}/models",
    f"{project_name}/tests",
    f"{project_name}/docker",
    f"{project_name}/data",
    f"{project_name}/notebooks"
]

for folder in folders_to_create:
    os.makedirs(folder, exist_ok=True)

print("Структура папок успешно создана на Google Диске.")

# 2. Загружаем данные
# Файл UCI_Credit_Card.csv должен находиться в корне Colab (/content/)
data_path = '/content/UCI_Credit_Card.csv'
df = pd.read_csv(data_path)

# Удаляем столбец ID, он не несет полезной информации для модели
df = df.drop('ID', axis=1)
# Переименовываем целевую переменную для удобства
df = df.rename(columns={'default.payment.next.month': 'target'})

# 3. Анализ дисбаланса классов
print("\n--- Анализ дисбаланса классов ---")
class_counts = df['target'].value_counts()
class_ratios = df['target'].value_counts(normalize=True) * 100
print(f"Класс 0 (Нет дефолта): {class_counts[0]} ({class_ratios[0]:.2f}%)")
print(f"Класс 1 (Дефолт):      {class_counts[1]} ({class_ratios[1]:.2f}%)")
print("Вывод: Классы несбалансированы. Дефолтов меньше, поэтому будем использовать class_weight='balanced'.")

# 4. Разделяем данные на признаки (X) и целевую переменную (y)
X = df.drop('target', axis=1)
y = df['target']

# Разделяем данные на обучающую (80%) и тестовую (20%) выборки
# stratify=y гарантирует сохранение пропорции классов в обеих выборках
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nРазмер обучающей выборки: {X_train.shape[0]} примеров")
print(f"Размер тестовой выборки: {X_test.shape[0]} примеров")

# 5. Определение типов признаков
# Непрерывные числовые признаки (требуют масштабирования)
numeric_features = ['LIMIT_BAL', 'AGE', 'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3',
                    'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1', 'PAY_AMT2',
                    'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6']

# Категориальные и порядковые признаки (требуют кодирования)
categorical_features = ['SEX', 'EDUCATION', 'MARRIAGE', 'PAY_0', 'PAY_2',
                        'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6']

# 6. Создание пайплайна предобработки
# ColumnTransformer применяет разные преобразования к разным столбцам
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        # handle_unknown='ignore' защитит модель от падения, если придет новая категория
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
    ])

# 7. Создание полного пайплайна (предобработка + модель)
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    # class_weight='balanced' автоматически увеличивает штраф за ошибки на редком классе
    ('classifier', LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000))
])

# 8. Обучение пайплайна
print("\n--- Обучение модели ---")
pipeline.fit(X_train, y_train)

# 9. Оценка качества модели
y_pred = pipeline.predict(X_test)

print("\nМетрики качества модели:")
print(f"Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred):.4f}")
print(f"F1-score:  {f1_score(y_test, y_pred):.4f}")

print("\nПодробный отчет о классификации:")
print(classification_report(y_test, y_pred, target_names=['No Default (0)', 'Default (1)']))

# 10. Сохранение пайплайна в папку models на Google Диске
pipeline_path = f"{project_name}/models/pipeline_v1.pkl"
joblib.dump(pipeline, pipeline_path)
print(f"\nПайплайн сохранен: {pipeline_path}")

Структура папок успешно создана на Google Диске.

--- Анализ дисбаланса классов ---
Класс 0 (Нет дефолта): 23364 (77.88%)
Класс 1 (Дефолт):      6636 (22.12%)
Вывод: Классы несбалансированы. Дефолтов меньше, поэтому будем использовать class_weight='balanced'.

Размер обучающей выборки: 24000 примеров
Размер тестовой выборки: 6000 примеров

--- Обучение модели ---

Метрики качества модели:
Accuracy:  0.7735
Precision: 0.4893
Recall:    0.5531
F1-score:  0.5193

Подробный отчет о классификации:
                precision    recall  f1-score   support

No Default (0)       0.87      0.84      0.85      4673
   Default (1)       0.49      0.55      0.52      1327

      accuracy                           0.77      6000
     macro avg       0.68      0.69      0.69      6000
  weighted avg       0.78      0.77      0.78      6000


Пайплайн сохранен: /content/drive/MyDrive/Ucheba/Input ML/credit-card-ml-deployment/models/pipeline_v1.pkl


In [5]:
# Шаг 2. Создание обработчика модели
# Код для файла model_handler.py
model_handler_code = """
import joblib
import os
import pandas as pd

class ModelHandler:
    def __init__(self, pipeline_path):
        self.pipeline_path = pipeline_path
        self.pipeline = None

    def load_model(self):
        if not os.path.exists(self.pipeline_path):
            raise FileNotFoundError(f"Файл пайплайна не найден: {self.pipeline_path}")
        self.pipeline = joblib.load(self.pipeline_path)

    def predict(self, features_df):
        if self.pipeline is None:
            raise RuntimeError("Пайплайн не загружен. Вызовите load_model()")

        # Передаем DataFrame напрямую в пайплайн.
        # ColumnTransformer сам обработает числовые и категориальные признаки.
        prediction = self.pipeline.predict(features_df)[0]
        probability = self.pipeline.predict_proba(features_df)[0][1]

        return int(prediction), float(probability)
"""

# Записываем код в файл на Google Диске
file_path = f"{project_name}/app/model_handler.py"
with open(file_path, "w", encoding="utf-8") as f:
    f.write(model_handler_code)

print(f"Файл создан: {file_path}")

Файл создан: /content/drive/MyDrive/Ucheba/Input ML/credit-card-ml-deployment/app/model_handler.py


In [8]:
# Шаг 3. Создание Flask-приложения (ИСПРАВЛЕННАЯ ВЕРСИЯ)
api_code = """
from flask import Flask, request, jsonify
import sys
import os
import pandas as pd
import logging
import json
import time
from datetime import datetime

# Добавляем путь к папке app для импорта model_handler
sys.path.append(os.path.dirname(__file__))
from model_handler import ModelHandler

app = Flask(__name__)

# === Настройка JSON-логирования ===
api_logger = logging.getLogger('api_logger')
api_logger.setLevel(logging.INFO)

os.makedirs('logs', exist_ok=True)
file_handler = logging.FileHandler('logs/api_requests.log', encoding='utf-8')
file_handler.setLevel(logging.INFO)

class JsonFormatter(logging.Formatter):
    def format(self, record):
        log_record = {
            "timestamp": datetime.utcnow().isoformat(),
            "level": record.levelname,
            "message": record.getMessage()
        }
        if hasattr(record, 'extra_data'):
            log_record.update(record.extra_data)
        return json.dumps(log_record, ensure_ascii=False)

file_handler.setFormatter(JsonFormatter())
api_logger.addHandler(file_handler)

console_handler = logging.StreamHandler()
console_handler.setLevel(logging.INFO)
console_handler.setFormatter(JsonFormatter())
api_logger.addHandler(console_handler)

# === Загрузка модели ===
BASE_DIR = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
model_handler = ModelHandler(
    pipeline_path=os.path.join(BASE_DIR, "models", "pipeline_v1.pkl")
)
model_handler.load_model()


def validate_input(data):
    '''Валидация входных данных клиента.'''
    numeric_features = ['LIMIT_BAL', 'AGE', 'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3',
                        'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1', 'PAY_AMT2',
                        'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6']

    for feature in numeric_features:
        if feature not in data:
            return False, f"Отсутствует признак: {feature}"
        if not isinstance(data[feature], (int, float)):
            return False, f"Признак {feature} должен быть числом"
        if data[feature] < 0:
            return False, f"Признак {feature} не может быть отрицательным"

    if data['AGE'] < 18 or data['AGE'] > 100:
        return False, "AGE должен быть в диапазоне от 18 до 100"

    if data.get('SEX') not in [1, 2]:
        return False, "SEX должен быть 1 (male) или 2 (female)"

    if data.get('EDUCATION') not in [1, 2, 3, 4, 5, 6]:
        return False, "EDUCATION должен быть от 1 до 6"

    if data.get('MARRIAGE') not in [1, 2, 3]:
        return False, "MARRIAGE должен быть 1, 2 или 3"

    pay_features = ['PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6']
    for feature in pay_features:
        if feature not in data:
            return False, f"Отсутствует признак: {feature}"
        if data[feature] not in [-1, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9]:
            return False, f"Признак {feature} должен быть от -1 до 9"

    return True, "OK"


@app.route('/predict', methods=['POST'])
def predict():
    '''Эндпоинт предсказания дефолта с JSON-логированием.'''
    start_time = time.time()
    client_ip = request.remote_addr

    try:
        data = request.get_json()
        if data is None:
            log_extra = {
                "endpoint": "/predict",
                "client_ip": client_ip,
                "status": "error",
                "error": "Invalid JSON"
            }
            api_logger.info("Invalid request", extra={'extra_data': log_extra})
            return jsonify({"error": "Неверный формат данных, ожидается JSON"}), 400

        is_valid, message = validate_input(data)
        if not is_valid:
            log_extra = {
                "endpoint": "/predict",
                "client_ip": client_ip,
                "status": "validation_failed",
                "error": message
            }
            api_logger.info("Validation failed", extra={'extra_data': log_extra})
            return jsonify({"error": message}), 400

        features_df = pd.DataFrame([data])
        prediction, probability = model_handler.predict(features_df)

        response_time_ms = int((time.time() - start_time) * 1000)

        log_extra = {
            "endpoint": "/predict",
            "client_ip": client_ip,
            "status": "success",
            "prediction": prediction,
            "probability": round(probability, 4),
            "model_version": "v2_pipeline",
            "response_time_ms": response_time_ms
        }
        api_logger.info("Prediction made", extra={'extra_data': log_extra})

        return jsonify({
            "prediction": prediction,
            "probability": round(probability, 4),
            "model_version": "v2_pipeline"
        }), 200

    except Exception as e:
        log_extra = {
            "endpoint": "/predict",
            "client_ip": client_ip,
            "status": "internal_error",
            "error": str(e)
        }
        api_logger.error("Internal error", extra={'extra_data': log_extra})
        return jsonify({"error": str(e)}), 500


@app.route('/health', methods=['GET'])
def health():
    '''Эндпоинт проверки работоспособности.'''
    return jsonify({"status": "healthy"}), 200


if __name__ == '__main__':
    app.run(host='0.0.0.0', port=5000, debug=False)
"""

file_path = f"{project_name}/app/api.py"
with open(file_path, "w", encoding="utf-8") as f:
    f.write(api_code)

init_path = f"{project_name}/app/__init__.py"
with open(init_path, "w") as f:
    f.write("")

print(f"Файлы Flask-сервиса успешно созданы: {file_path}")

Файлы Flask-сервиса успешно созданы: /content/drive/MyDrive/Ucheba/Input ML/credit-card-ml-deployment/app/api.py


In [9]:
# Шаг 4. Создание конфигурационных файлов и документации
# 1. requirements.txt
requirements_content = """flask==3.0.0
scikit-learn==1.6.1
pandas==2.1.4
numpy==1.26.2
joblib==1.3.2
gunicorn==21.2.0
scipy==1.16.3
pytest==7.4.3
"""
with open(f"{project_name}/requirements.txt", "w") as f:
    f.write(requirements_content)

# 2. Dockerfile
dockerfile_content = """FROM python:3.11-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY app/ ./app/
COPY models/ ./models/
EXPOSE 5000
CMD ["gunicorn", "--bind", "0.0.0.0:5000", "--workers", "4", "app.api:app"]
"""
with open(f"{project_name}/docker/Dockerfile", "w") as f:
    f.write(dockerfile_content)


print("Конфигурационные файлы созданы.")

Конфигурационные файлы созданы.


In [10]:
# Шаг 5. Тестирование сервиса внутри Colab
import threading
import time
import requests
import sys
import os

# Переходим в папку app, чтобы импорты работали корректно
os.chdir(f"{project_name}/app")
sys.path.insert(0, f"{project_name}/app")

# Импортируем наше Flask-приложение
from api import app

# Функция для запуска сервера в фоновом потоке
def run_flask():
    # use_reloader=False отключает автоматическую перезагрузку, которая мешает фоновому запуску
    app.run(host='0.0.0.0', port=5000, debug=False, use_reloader=False)

thread = threading.Thread(target=run_flask)
thread.daemon = True
thread.start()

# Ждем 3 секунды, пока сервер запустится
time.sleep(3)

# Тест 1: Проверка здоровья (GET /health)
print("--- Тест 1: Эндпоинт /health ---")
response_health = requests.get('http://localhost:5000/health')
print(f"Статус: {response_health.status_code}")
print(f"Ответ: {response_health.json()}\n")

# Тест 2: Успешное предсказание (POST /predict)
print("--- Тест 2: Успешное предсказание (валидные данные) ---")
test_data_valid = {
    "LIMIT_BAL": 20000, "SEX": 2, "EDUCATION": 2, "MARRIAGE": 1, "AGE": 24,
    "PAY_0": 2, "PAY_2": 2, "PAY_3": -1, "PAY_4": -1, "PAY_5": -1, "PAY_6": -1,
    "BILL_AMT1": 3913, "BILL_AMT2": 3102, "BILL_AMT3": 689, "BILL_AMT4": 0,
    "BILL_AMT5": 0, "BILL_AMT6": 0, "PAY_AMT1": 689, "PAY_AMT2": 0,
    "PAY_AMT3": 0, "PAY_AMT4": 0, "PAY_AMT5": 0, "PAY_AMT6": 0
}
response = requests.post('http://localhost:5000/predict', json=test_data_valid)
print(f"Статус: {response.status_code}")
print(f"Ответ: {response.json()}\n")

# Тест 3: Ошибка валидации (неверный возраст)
print("--- Тест 3: Ошибка валидации (неверный возраст 150) ---")
test_data_invalid = test_data_valid.copy()
test_data_invalid['AGE'] = 150
response = requests.post('http://localhost:5000/predict', json=test_data_invalid)
print(f"Статус: {response.status_code}")
print(f"Ответ: {response.json()}\n")

# Тест 4: Ошибка валидации (неверный пол)
print("--- Тест 4: Ошибка валидации (неверный пол 3) ---")
test_data_invalid_2 = test_data_valid.copy()
test_data_invalid_2['SEX'] = 3
response = requests.post('http://localhost:5000/predict', json=test_data_invalid_2)
print(f"Статус: {response.status_code}")
print(f"Ответ: {response.json()}")

# Возвращаемся в корневую папку Colab
os.chdir('/content')

 * Serving Flask app 'api'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [10/Jun/2026 07:45:47] "GET /health HTTP/1.1" 200 -
{"timestamp": "2026-06-10T07:45:47.172875", "level": "INFO", "message": "Prediction made", "endpoint": "/predict", "client_ip": "127.0.0.1", "status": "success", "prediction": 1, "probability": 0.8546, "model_version": "v2_pipeline", "response_time_ms": 13}
INFO:api_logger:Prediction made
INFO:werkzeug:127.0.0.1 - - [10/Jun/2026 07:45:47] "POST /predict HTTP/1.1" 200 -
{"timestamp": "2026-06-10T07:45:47.182162", "level": "INFO", "message": "Validation failed", "endpoint": "/predict", "client_ip": "127.0.0.1", "status": "validation_failed", "error": "AGE должен быть в диапазоне от 18 до 100"}
INFO:api_logger:Validation failed
INFO:we

--- Тест 1: Эндпоинт /health ---
Статус: 200
Ответ: {'status': 'healthy'}

--- Тест 2: Успешное предсказание (валидные данные) ---
Статус: 200
Ответ: {'model_version': 'v2_pipeline', 'prediction': 1, 'probability': 0.8546}

--- Тест 3: Ошибка валидации (неверный возраст 150) ---
Статус: 400
Ответ: {'error': 'AGE должен быть в диапазоне от 18 до 100'}

--- Тест 4: Ошибка валидации (неверный пол 3) ---
Статус: 400
Ответ: {'error': 'SEX должен быть 1 (male) или 2 (female)'}


In [11]:
# Код для файла tests/test_api.py
test_api_code = """
import pytest
import sys
import os
import json

# Добавляем путь к корню проекта, чтобы корректно импортировать app
# Это необходимо для корректной работы путей при запуске pytest из корня репозитория
sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname(__file__), '..')))

from app.api import app

@pytest.fixture
def client():
    \"\"\"Фикстура pytest для создания тестового клиента Flask.\"\"\"
    app.config['TESTING'] = True
    # Отключаем JSON-логирование в файл во время тестов, чтобы не засорять диск
    # (в реальном проекте можно настроить отдельный логгер для тестов)
    with app.test_client() as client:
        yield client

def test_health_endpoint(client):
    \"\"\"Тест 1: Проверка эндпоинта /health.\"\"\"
    response = client.get('/health')
    assert response.status_code == 200

    data = json.loads(response.data)
    assert data['status'] == 'healthy'

def test_predict_success(client):
    \"\"\"Тест 2: Успешное предсказание с валидными данными.\"\"\"
    test_data = {
        "LIMIT_BAL": 20000, "SEX": 2, "EDUCATION": 2, "MARRIAGE": 1, "AGE": 24,
        "PAY_0": 2, "PAY_2": 2, "PAY_3": -1, "PAY_4": -1, "PAY_5": -1, "PAY_6": -1,
        "BILL_AMT1": 3913, "BILL_AMT2": 3102, "BILL_AMT3": 689, "BILL_AMT4": 0,
        "BILL_AMT5": 0, "BILL_AMT6": 0, "PAY_AMT1": 689, "PAY_AMT2": 0,
        "PAY_AMT3": 0, "PAY_AMT4": 0, "PAY_AMT5": 0, "PAY_AMT6": 0
    }

    response = client.post('/predict', json=test_data)
    assert response.status_code == 200

    data = json.loads(response.data)
    assert 'prediction' in data
    assert 'probability' in data
    assert data['model_version'] == 'v2_pipeline'
    assert isinstance(data['prediction'], int)
    assert 0.0 <= data['probability'] <= 1.0

def test_predict_validation_error_age(client):
    \"\"\"Тест 3: Ошибка валидации (некорректный возраст).\"\"\"
    test_data = {
        "LIMIT_BAL": 20000, "SEX": 2, "EDUCATION": 2, "MARRIAGE": 1, "AGE": 150, # Невалидный возраст
        "PAY_0": 2, "PAY_2": 2, "PAY_3": -1, "PAY_4": -1, "PAY_5": -1, "PAY_6": -1,
        "BILL_AMT1": 3913, "BILL_AMT2": 3102, "BILL_AMT3": 689, "BILL_AMT4": 0,
        "BILL_AMT5": 0, "BILL_AMT6": 0, "PAY_AMT1": 689, "PAY_AMT2": 0,
        "PAY_AMT3": 0, "PAY_AMT4": 0, "PAY_AMT5": 0, "PAY_AMT6": 0
    }

    response = client.post('/predict', json=test_data)
    assert response.status_code == 400

    data = json.loads(response.data)
    assert 'error' in data
    assert 'AGE' in data['error']

def test_predict_validation_error_sex(client):
    \"\"\"Тест 4: Ошибка валидации (некорректный пол).\"\"\"
    test_data = {
        "LIMIT_BAL": 20000, "SEX": 3, # Невалидный пол
        "EDUCATION": 2, "MARRIAGE": 1, "AGE": 24,
        "PAY_0": 2, "PAY_2": 2, "PAY_3": -1, "PAY_4": -1, "PAY_5": -1, "PAY_6": -1,
        "BILL_AMT1": 3913, "BILL_AMT2": 3102, "BILL_AMT3": 689, "BILL_AMT4": 0,
        "BILL_AMT5": 0, "BILL_AMT6": 0, "PAY_AMT1": 689, "PAY_AMT2": 0,
        "PAY_AMT3": 0, "PAY_AMT4": 0, "PAY_AMT5": 0, "PAY_AMT6": 0
    }

    response = client.post('/predict', json=test_data)
    assert response.status_code == 400

    data = json.loads(response.data)
    assert 'error' in data
    assert 'SEX' in data['error']
"""

# Записываем код в файл на Google Диске
file_path = f"{project_name}/tests/test_api.py"
with open(file_path, "w", encoding="utf-8") as f:
    f.write(test_api_code)

print(f"Файл автоматических тестов создан: {file_path}")

Файл автоматических тестов создан: /content/drive/MyDrive/Ucheba/Input ML/credit-card-ml-deployment/tests/test_api.py
